In [12]:
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from datasets import load_dataset

# load the dataset
ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")

# load bi-encoder and cross-encoder models
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print(len(ds))
print(ds[0]["title"])
print(ds[0]["text"][:500])
print("Models loaded!!")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14018.84it/s]


241787
April
April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

The Month 

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, Septembe
Models loaded!!


In [13]:
# split articles into passages of 200 words
passages = []
for article in ds:
    words = article["text"].split()
    for i in range(0, len(words), 200):
        chunk = " ".join(words[i:i+200])
        passages.append({"title": article["title"], "text": chunk})

print(len(passages))

368350


In [14]:
embeddings = bi_encoder.encode(
    [p["text"] for p in passages],
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(embeddings.shape)

Batches: 100%|██████████| 1439/1439 [18:54<00:00,  1.27it/s]


(368350, 384)


In [15]:
# build the faiss index, lets you search for similar passages using cosine similarity
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1]) # index uses inner product (cosine similarity) for search
index.add(embeddings) # loads all passage embeddings into the index
print(index.ntotal) 

368350


In [16]:
# save index, you can load it later without having to recompute the embeddings; when u reopen file, need run all the code again :(
import pickle

faiss.write_index(index, "index.faiss") # saves the faiss index to disk
with open("passages.pkl", "wb") as f: # saves the passage texts to passages.pkl (so you can show words to the user)
    pickle.dump(passages, f)

print("Saved!!")

Saved!!


In [17]:
# so we dont rerun the code above!
import faiss, pickle

index = faiss.read_index("index.faiss")
with open("passages.pkl", "rb") as f:
    passages = pickle.load(f)

In [18]:
# search fxn, takes a query and returns the top 5 most relevant passages
def search(query, top_k):
    # layer 1: bi-encoder search to get top k passages
    query_embedding = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    bi_scores, indices = index.search(query_embedding, top_k) # search the index for the top k most similar passages
    
    # layer 2: cross-encoder re-ranking to get the top 5 passages
    cross_inputs = [[query, passages[i]["text"]] for i in indices[0]]
    cross_scores = cross_encoder.predict(cross_inputs)
    top_indices = sorted(range(len(cross_scores)), key=lambda i: cross_scores[i], reverse=True)[:5] # get indices of top 5 passages
    return [passages[indices[0][i]] for i in top_indices]

In [24]:
# test
results = search("dachshund", top_k=10)
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['title']}")
    print(r['text'][:200])
    print()

[1] Dachshund
The dachshund is a breed of dog. It is usually short (small in height), but long from front to back and with short legs. “Dachshund” is a German word that is used in English, and it means “badger dog.

[2] Dachshund
found drawings and engravings from ancient Egypt that show hunting dogs with short legs. But the dachshund of today was bred in Europe, and it has parts of German, French, and English hounds and terri

[3] Dachshund
front to back than the other two. Size A fully-grown dachshund weighs between 16 and 28 lb. (7 to 12.7 kg), while the miniature dachshund may weigh less than 11 lb. (5 kg). The writer H. L. Mencken sa

[4] Dachshund
to their owners (devoted to their owners, or very interested in their owners). In fact, they can be much more loyal than other kinds of dogs. But wire-haired dachshunds often do not show that kind of 

[5] Dachsen
Dachsen is a municipality in the district of Andelfingen in the canton of Zurich in Switzerland. References Municipalities i